# SoulX-FlashHead: Talking Head Generator (Kaggle Version)

### ⚠️ **REQUIRED SETTINGS (Right Sidebar)**
1. **Accelerator**: Select **T4 GPU** (Standard T4 or T4 x2). 
   - *Note: **Tesla P100 is NOT supported** by modern PyTorch libraries used here.*
2. **Internet**: Switch **ON**. *(Required to download models and dependencies)*

In [ ]:
# @title 1. Setup Environment & Hardware Check
import os
import torch
import socket

# 1. Hardware Compatibility Check
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    print(f"Detected GPU: {gpu_name} (CUDA Capability {capability[0]}.{capability[1]})")
    
    if capability[0] < 7:
        print("\033[91m❌ FATAL ERROR: Incompatible GPU Detected!\033[0m")
        print(f"The {gpu_name} (P100) is too old for this project.")
        print("\033[93m➜ ACTION: Please switch to 'T4 GPU' in the Kaggle sidebar settings.\033[0m")
        # Not raising SystemExit yet, to let the user see the print, but further steps will likely fail.
else:
    print("\033[91m❌ ERROR: No GPU found. Please enable 'T4 GPU' in the sidebar.\033[0m")

# 2. Internet Connectivity Check
try:
    socket.gethostbyname("github.com")
except socket.gaierror:
    print("\033[91m❌ ERROR: Internet is DISABLED. Please enable 'Internet' in the Kaggle settings.\033[0m")
    raise SystemExit("Stopping execution: No Internet connection.")

# 3. Base Paths
BASE_PATH = "/kaggle/working"
print(f"Base Path: {BASE_PATH}")

# 4. Install Dependencies (Forcing latest transformers to fix MT5Tokenizer error)
!git clone https://github.com/Soul-AILab/SoulX-FlashHead.git
%cd {BASE_PATH}/SoulX-FlashHead

!apt-get install -y ffmpeg -q
!pip install -q xformers
# Force upgrade transformers and sentencepiece
!pip install -q -U transformers sentencepiece
!pip install -q "gradio>=5.0.0" diffusers accelerate tqdm imageio easydict ftfy imageio-ffmpeg scikit-image loguru pyloudnorm decord librosa flask huggingface_hub
!pip install -q "xfuser>=0.4.3" yunchang distvae beautifulsoup4 einops

# Fix for face detection (replaces broken mediapipe with OpenCV)
os.makedirs("flash_head/utils", exist_ok=True)
with open("flash_head/utils/cpu_face_handler.py", "w") as f:
    f.write(f'''import cv2\nimport numpy as np\nimport os\nfrom typing import Tuple, List\n\nclass CPUFaceHandler:\n    def __init__(self, model_selection: int = 1, min_detection_confidence: float = 0.0):\n        proto = "{BASE_PATH}/SoulX-FlashHead/deploy.prototxt"\n        caffe = "{BASE_PATH}/SoulX-FlashHead/res10_300x300_ssd_iter_140000.caffemodel"\n        self.use_dnn = False\n        import urllib.request\n        if not os.path.exists(proto):\n            try: urllib.request.urlretrieve("https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt", proto)\n            except: pass\n        if not os.path.exists(caffe):\n            try: urllib.request.urlretrieve("https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel", caffe)\n            except: pass\n        if os.path.exists(proto) and os.path.exists(caffe):\n            self.net = cv2.dnn.readNetFromCaffe(proto, caffe)\n            self.use_dnn = True\n        else:\n            cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"\n            self.cascade = cv2.CascadeClassifier(cascade_path)\n\n    def detect(self, image: np.ndarray) -> Tuple[List, List]:\n        bboxs, scores = [], []\n        img_h, img_w = image.shape[:2]\n        if self.use_dnn:\n            blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300), (104.0, 177.0, 123.0))\n            self.net.setInput(blob)\n            detections = self.net.forward()\n            for i in range(detections.shape[2]):\n                confidence = float(detections[0, 0, i, 2])\n                if confidence > 0.5:\n                    bboxs.append([float(detections[0, 0, i, 3]), float(detections[0, 0, i, 4]), float(detections[0, 0, i, 5]), float(detections[0, 0, i, 6])])\n                    scores.append(confidence)\n        else:\n            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)\n            faces = self.cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))\n            for (x, y, w, h) in faces:\n                bboxs.append([x/img_w, y/img_h, (x+w)/img_w, (y+h)/img_h]); scores.append(1.0)\n        return bboxs, scores\n\n    def __call__(self, image: np.ndarray) -> Tuple[List, List]:\n        return self.detect(image)\n''')

# SAFE PATCH for facecrop.py to ALWAYS save coordinates for 9:16 compositing
with open("flash_head/utils/facecrop.py", "r") as f:
    fc_lines = f.readlines()

if "import json" not in "".join(fc_lines):
    fc_lines.insert(7, "import json\n")

with open("flash_head/utils/facecrop.py", "w") as f:
    for line in fc_lines:
        if "crop_face = face_image.crop(scaled_bbox)" in line:
            indent = line[:line.find("crop_face")]
            f.write(f"{indent}with open('/tmp/bbox.json', 'w') as bf: json.dump({{ 'bbox': scaled_bbox, 'img_w': img_w, 'img_h': img_h }}, bf)\n")
        f.write(line)

print("✅ Setup complete")

In [ ]:
# @title 2. Download Models
from huggingface_hub import snapshot_download
import os

os.makedirs(f"{BASE_PATH}/models", exist_ok=True)

print("📥 Downloading SoulX-FlashHead (1.3B)... may take a few mins")
snapshot_download(
    repo_id="Soul-AILab/SoulX-FlashHead-1_3B",
    local_dir=f"{BASE_PATH}/models/SoulX-FlashHead-1_3B",
    local_dir_use_symlinks=False
)

print("📥 Downloading wav2vec2-base-960h...")
snapshot_download(
    repo_id="facebook/wav2vec2-base-960h",
    local_dir=f"{BASE_PATH}/models/wav2vec2-base-960h",
    local_dir_use_symlinks=False
)

print("✅ Models downloaded successfully!")

In [ ]:
# @title 3. Start Launch App (Auto-9:16 Composite)
import os
import re

with open('gradio_app.py', 'r') as f: code = f.read()

# Set defaults dynamically
code = code.replace('value="models/SoulX-FlashHead-1_3B"', f'value="{BASE_PATH}/models/SoulX-FlashHead-1_3B"')
code = code.replace('value="models/wav2vec2-base-960h"', f'value="{BASE_PATH}/models/wav2vec2-base-960h"')
code = code.replace('value="lite")', 'value="lite")') # Just in case
code = code.replace('value=False,\n                            info="Enable face detection', 'value=True,\n                            info="Enable face detection')
code = code.replace('app.launch()', 'app.launch(share=True, debug=True)')

# Robust 9:16 Compositing Logic
composite_func = """
def _composite_to_original(video_path, image_path):
    import cv2, json, numpy as np, subprocess, os
    if not os.path.exists('/tmp/bbox.json'): 
        print("DEBUG: /tmp/bbox.json not found. Make sure 'Use Face Crop' is enabled.")
        return video_path
    with open('/tmp/bbox.json', 'r') as f: data = json.load(f)
    x1, y1, x2, y2 = data['bbox']
    w, h = data['img_w'], data['img_h']
    print(f"DEBUG: Image resolution is {w}x{h}, BBox is {x1,y1,x2,y2}")
    
    # If original is square or horizontal, skip compositing to keep things simple
    if w >= h * 0.9: 
        print("DEBUG: Image is square or horizontal, skipping compositing.")
        return video_path
        
    orig = cv2.imread(image_path)
    cap = cv2.VideoCapture(video_path)
    temp_vid = video_path.replace('.mp4', '_tmp916.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(temp_vid, fourcc, 25.0, (w, h))
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_resized = cv2.resize(frame, (int(round(x2-x1)), int(round(y2-y1))))
        canvas = orig.copy()
        canvas[int(round(y1)):int(round(y2)), int(round(x1)):int(round(x2))] = frame_resized
        out.write(canvas)
    cap.release(); out.release()
    
    final_916 = video_path.replace('.mp4', '_final916.mp4')
    subprocess.run(['ffmpeg', '-y', '-i', temp_vid, '-i', video_path, '-c:v', 'libx264', '-map', '0:v', '-map', '1:a', '-shortest', final_916], capture_output=True)
    
    if os.path.exists(final_916):
        os.replace(final_916, video_path)
        if os.path.exists(temp_vid): os.remove(temp_vid)
        print(f"✅ Successfully composited to 9:16 vertical video!")
    return video_path
"""

code = "import json, cv2, subprocess, numpy as np\n" + composite_func + code
code = code.replace("return final_video_path", "return _composite_to_original(final_video_path, cond_image)")
code = code.replace("return save_path", "return _composite_to_original(save_path, cond_image)")

with open('gradio_app_916.py', 'w') as f: f.write(code)

print("🚀 Launching with 9:16 Auto-Support... Ensure 'Use Face Crop' is CHECKED!")
!python gradio_app_916.py